In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Cell 1 — GPU allocator

In [2]:
import os
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
print("GPU allocator set.")

GPU allocator set.


Cell 2 — Install dependencies

In [3]:
import subprocess, sys

def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *pkgs, '-q'], check=True)

pip('mne', 'awscli', 'tensorflow', 'scikit-learn',
    'numpy', 'pandas', 'scipy', 'matplotlib', 'seaborn')

print('All dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 27.2 MB/s eta 0:00:00
All dependencies installed.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
aiobotocore 3.3.0 requires botocore<1.42.71,>=1.42.62, but you have botocore 1.42.96 which is incompatible.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


Cell 3 — Paths

In [4]:
import os
from pathlib import Path

if os.path.exists('/content'):
    ENV, BASE = 'colab', Path('/content/sir-eegnet')
elif os.path.exists('/kaggle/working'):
    ENV, BASE = 'kaggle', Path('/kaggle/working/sir-eegnet')
else:
    ENV, BASE = 'local', Path.cwd().parent

DATA_DIR     = BASE / 'data'
DS_DIR       = DATA_DIR / 'ds004504'
DERIV_DIR    = DS_DIR / 'derivatives'
PARTICIPANTS = DS_DIR / 'participants.tsv'
RESULTS_DIR  = BASE / 'results'
MODELS_DIR   = BASE / 'saved_models'

for d in [DATA_DIR, DS_DIR, DERIV_DIR, RESULTS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Environment : {ENV}')
print(f'Base        : {BASE}')

Environment : colab
Base        : /content/sir-eegnet


Cell 4 — Constants

In [5]:
import numpy as np
import pandas as pd
np.random.seed(42)

SFREQ          = 500.0
N_CHANNELS     = 19
EPOCH_DURATION = 4.0
EPOCH_OVERLAP  = 0.5
EPOCH_SAMPLES  = int(EPOCH_DURATION * SFREQ)   # 2000

GROUP_MAP   = {'A': 0, 'F': 1, 'C': 2}
LABEL_NAMES = {0: 'AD', 1: 'FTD', 2: 'HC'}

# EEGNet — Lawhern et al. 2018
# kernLength = SFREQ/2 as recommended in paper
KERN_LENGTH = 250
F1          = 8
D           = 2
DROPOUT     = 0.5

# Training — tuned for overnight Kaggle batch run
BATCH_SIZE   = 64    # larger = faster GPU utilization
MAX_EPOCHS   = 50    # sufficient with early stopping
PATIENCE     = 10
VAL_FRAC     = 0.10
MAX_EPOCHS_PER_SUBJECT = 150  # cap epochs per subject to control memory

print(f'Epoch shape  : ({N_CHANNELS}, {EPOCH_SAMPLES}, 1)')
print(f'kernLength   : {KERN_LENGTH}')
print(f'Batch size   : {BATCH_SIZE}')
print(f'Max epochs   : {MAX_EPOCHS} (with early stopping, patience={PATIENCE})')

Epoch shape  : (19, 2000, 1)
kernLength   : 250
Batch size   : 64
Max epochs   : 50 (with early stopping, patience=10)


Cell 5 — Download dataset

In [6]:
import subprocess

def download_ds004504(deriv_dir, participants_path):
    set_files = list(deriv_dir.rglob('*.set'))
    if participants_path.exists() and len(set_files) >= 80:
        print(f'Dataset already present ({len(set_files)} .set files). Skipping.')
        return
    print('Downloading ds004504 from OpenNeuro S3 (~2.7 GB)...')
    r1 = subprocess.run([
        'aws', 's3', 'sync',
        's3://openneuro.org/ds004504/derivatives/', str(deriv_dir),
        '--no-sign-request', '--region', 'us-east-1'
    ], capture_output=True, text=True)
    if r1.returncode != 0:
        raise RuntimeError(f'Download failed:\n{r1.stderr[-600:]}')
    r2 = subprocess.run([
        'aws', 's3', 'cp',
        's3://openneuro.org/ds004504/participants.tsv', str(participants_path),
        '--no-sign-request', '--region', 'us-east-1'
    ], capture_output=True, text=True)
    if r2.returncode != 0:
        raise RuntimeError(f'TSV failed:\n{r2.stderr[-600:]}')
    print('Download complete.')

download_ds004504(DERIV_DIR, PARTICIPANTS)

Download complete.


Cell 6 — Load participants

In [7]:
participants = pd.read_csv(PARTICIPANTS, sep='\t')
participants['label'] = participants['Group'].map(GROUP_MAP)
participants = participants.dropna(subset=['label'])
participants['label'] = participants['label'].astype(int)

print(f'Total subjects: {len(participants)}')
for label, name in LABEL_NAMES.items():
    n = (participants['label'] == label).sum()
    print(f'  {name}: {n}')

Total subjects: 88
  AD: 36
  FTD: 23
  HC: 29


Cell 7 — Build subject metadata index 

In [8]:
import mne
mne.set_log_level('WARNING')

subject_meta = {}
failed = []

for _, row in participants.iterrows():
    sid   = row['participant_id']
    label = int(row['label'])
    fpath = DERIV_DIR / sid / 'eeg' / f'{sid}_task-eyesclosed_eeg.set'
    if not fpath.exists():
        failed.append(sid)
        continue
    subject_meta[sid] = {
        'label': label,
        'group': row['Group'],
        'mmse':  float(row.get('MMSE', np.nan)),
        'age':   float(row['Age']),
        'path':  fpath,
    }

print(f'Index built: {len(subject_meta)} subjects | Failed: {len(failed)}')
for lbl, name in LABEL_NAMES.items():
    n = sum(1 for d in subject_meta.values() if d['label'] == lbl)
    print(f'  {name}: {n}')

Index built: 88 subjects | Failed: 0
  AD: 36
  FTD: 23
  HC: 29


Cell 8 — EEG loading and epoching

In [9]:
def load_subject_epochs(sid, meta, sfreq=500.0, duration=4.0,
                         overlap=0.5, max_epochs=150):
    """
    Load one subject's EEG from disk and slice into epochs.
    max_epochs caps the number of epochs to control memory.
    Capping at 150 is methodologically fine — random subsample
    preserves class signal while keeping fold runtime predictable.
    """
    raw    = mne.io.read_raw_eeglab(str(meta['path']),
                                     preload=True, verbose=False)
    data   = raw.get_data()
    del raw

    ep_s   = int(duration * sfreq)
    step_s = int(ep_s * (1 - overlap))
    starts = list(range(0, data.shape[1] - ep_s + 1, step_s))
    epochs = np.stack([data[:, s:s + ep_s] for s in starts], axis=0).astype(np.float32)
    del data

    if len(epochs) > max_epochs:
        idx    = np.random.RandomState(42).choice(len(epochs), max_epochs, replace=False)
        epochs = epochs[idx]

    return epochs   # (n_epochs, 19, 2000)


# Quick verification
test_ep = load_subject_epochs('sub-001', subject_meta['sub-001'])
print(f'sub-001 epochs shape : {test_ep.shape}')
print(f'Value range          : [{test_ep.min():.4f}, {test_ep.max():.4f}]')
del test_ep

sub-001 epochs shape : (150, 19, 2000)
Value range          : [-0.0002, 0.0002]


Cell 9 — EEGNet architecture (Lawhern et al. 2018, original)

In [10]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, DepthwiseConv2D, SeparableConv2D,
    BatchNormalization, Activation, AveragePooling2D,
    Dropout, Flatten, Dense
)
from tensorflow.keras.constraints import max_norm

tf.random.set_seed(42)

def build_eegnet(nb_classes, n_channels=19, n_samples=2000,
                 dropout_rate=0.5, kern_length=250,
                 F1=8, D=2, norm_rate=0.25):
    """
    EEGNet — Lawhern et al. (2018), Journal of Neural Engineering.
    https://doi.org/10.1088/1741-2552/aace8c

    Input shape : (n_channels, n_samples, 1)
    Adaptations : n_channels=19, n_samples=2000, kern_length=250 (SFREQ/2)
    """
    F2     = F1 * D
    input1 = Input(shape=(n_channels, n_samples, 1))

    # Block 1: temporal convolution + depthwise spatial convolution
    x = Conv2D(F1, (1, kern_length), padding='same', use_bias=False)(input1)
    x = BatchNormalization()(x)
    x = DepthwiseConv2D((n_channels, 1), use_bias=False,
                         depth_multiplier=D,
                         depthwise_constraint=max_norm(1.))(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 4))(x)
    x = Dropout(dropout_rate)(x)

    # Block 2: separable convolution (temporal fusion)
    x = SeparableConv2D(F2, (1, 16), use_bias=False, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 8))(x)
    x = Dropout(dropout_rate)(x)

    x   = Flatten(name='flatten')(x)
    x   = Dense(nb_classes, name='dense',
                kernel_constraint=max_norm(norm_rate))(x)
    out = Activation('softmax', name='softmax')(x)

    return Model(inputs=input1, outputs=out)


# Architecture check
_m = build_eegnet(nb_classes=2, n_channels=N_CHANNELS, n_samples=EPOCH_SAMPLES)
print(f'EEGNet parameters : {_m.count_params():,}')
_m.summary()
del _m

2026-04-26 14:18:01.091625: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777213081.514106      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777213081.650263      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777213082.728035      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777213082.728079      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777213082.728083      22 computation_placer.cc:177] computation placer alr

EEGNet parameters : 4,962


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 19, 2000, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 19, 2000, 8)    │         2,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 19, 2000, 8)    │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 1, 2000, 16)    │           304 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1, 2000, 16)    │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 1, 2000, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 1, 500, 16)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 500, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d                │ (None, 1, 500, 16)     │           512 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1, 500, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 1, 500, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 1, 62, 16)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1, 62, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 992)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │         1,986 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax (Activation)            │ (None, 2)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,962 (19.38 KB)

 Trainable params: 4,882 (19.07 KB)

 Non-trainable params: 80 (320.00 B)

Cell 10 — Utility functions

In [11]:
import gc
import time
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix)


def fit_scaler_on_train(X_train):
    """
    Fit StandardScaler channel-wise on training epochs only.
    X_train: (N, C, S)
    Returns fitted scaler.
    NEVER call this on test data — only transform test, never fit.
    """
    N, C, S = X_train.shape
    X_2d    = X_train.transpose(0, 2, 1).reshape(-1, C)
    scaler  = StandardScaler().fit(X_2d)
    return scaler


def apply_scaler(X, scaler):
    """Transform X using a pre-fitted scaler. X: (N, C, S)"""
    N, C, S = X.shape
    X_2d    = X.transpose(0, 2, 1).reshape(-1, C)
    X_2d    = scaler.transform(X_2d)
    return X_2d.reshape(N, S, C).transpose(0, 2, 1).astype(np.float32)


def to_keras_input(X):
    """(N, C, S) -> (N, C, S, 1) for Keras Conv2D."""
    return X[:, :, :, np.newaxis]


def majority_vote(epoch_preds):
    """Epoch-level predictions -> single subject-level label."""
    return Counter(epoch_preds.tolist()).most_common(1)[0][0]


def compute_class_weights(y):
    """Inverse-frequency class weights."""
    classes, counts = np.unique(y, return_counts=True)
    total = len(y)
    return {int(c): total / (len(classes) * cnt)
            for c, cnt in zip(classes, counts)}


print('Utilities defined.')

Utilities defined.


Cell 11 — Full LOSO evaluation function

In [12]:
def run_eegnet_loso(subject_meta, n_classes, binary_remap=None,
                     val_frac=VAL_FRAC, max_epochs=MAX_EPOCHS,
                     patience=PATIENCE, batch_size=BATCH_SIZE,
                     lr=1e-3, max_ep_per_subject=MAX_EPOCHS_PER_SUBJECT,
                     verbose_every=5, label_names=None):
    """
    Strict Leave-One-Subject-Out evaluation for EEGNet.

    Leakage prevention — verified at every step:
    [1] Test subject excluded from all training data
    [2] StandardScaler fit on training epochs only, inside each fold
    [3] Validation subjects drawn from training pool only
    [4] Subject-level prediction = majority vote over test epochs
    [5] All reported metrics are subject-level

    binary_remap: e.g. {0:0, 2:1} keeps AD+HC only, remaps labels.
                  None = all classes used as-is.
    """
    if binary_remap is not None:
        sdata = {sid: dict(d, label=binary_remap[d['label']])
                 for sid, d in subject_meta.items()
                 if d['label'] in binary_remap}
    else:
        sdata = subject_meta

    all_sids    = list(sdata.keys())
    true_labels = []
    pred_labels = []
    histories   = []
    fold_times  = []

    n_folds = len(all_sids)
    print(f'LOSO: {n_folds} folds | {n_classes} classes')
    print(f'Epochs capped at {max_ep_per_subject}/subject | '
          f'batch={batch_size} | max_train_epochs={max_epochs} | patience={patience}')
    print(f'Estimated time: {n_folds * 40 / 60:.0f}–{n_folds * 60 / 60:.0f} min on T4 GPU')
    print()

    for fold_i, test_sid in enumerate(all_sids):

        t_fold = time.time()

        # ── 1. Subject split ───────────────────────────────────────────
        train_sids = [s for s in all_sids if s != test_sid]
        true_label = sdata[test_sid]['label']

        rng = np.random.RandomState(42 + fold_i)
        shuffled = train_sids.copy()
        rng.shuffle(shuffled)
        n_val    = max(1, int(len(shuffled) * val_frac))
        val_sids = shuffled[:n_val]
        tr_sids  = shuffled[n_val:]

        # ── 2. Load training epochs ────────────────────────────────────
        X_tr_list, y_tr_list = [], []
        for s in tr_sids:
            ep = load_subject_epochs(s, sdata[s],
                                      max_epochs=max_ep_per_subject)
            X_tr_list.append(ep)
            y_tr_list.extend([sdata[s]['label']] * len(ep))
            del ep
        X_tr = np.concatenate(X_tr_list, axis=0)
        y_tr = np.array(y_tr_list, dtype=np.int64)
        del X_tr_list

        # ── 3. Fit scaler on training data only ────────────────────────
        scaler = fit_scaler_on_train(X_tr)
        X_tr   = apply_scaler(X_tr, scaler)

        # ── 4. Load and scale validation subjects ──────────────────────
        X_val_list, y_val_list = [], []
        for s in val_sids:
            ep = load_subject_epochs(s, sdata[s],
                                      max_epochs=max_ep_per_subject)
            X_val_list.append(apply_scaler(ep, scaler))
            y_val_list.extend([sdata[s]['label']] * len(ep))
            del ep
        X_val = np.concatenate(X_val_list, axis=0)
        y_val = np.array(y_val_list, dtype=np.int64)
        del X_val_list

        # ── 5. Load and scale test subject ─────────────────────────────
        X_te = load_subject_epochs(test_sid, sdata[test_sid],
                                    max_epochs=max_ep_per_subject)
        X_te = apply_scaler(X_te, scaler)

        # ── 6. Reshape for Keras: (N, C, S, 1) ────────────────────────
        X_tr_k  = to_keras_input(X_tr)
        X_val_k = to_keras_input(X_val)
        X_te_k  = to_keras_input(X_te)

        # ── 7. Build model ─────────────────────────────────────────────
        tf.keras.backend.clear_session()
        model = build_eegnet(nb_classes=n_classes,
                              n_channels=N_CHANNELS,
                              n_samples=EPOCH_SAMPLES,
                              dropout_rate=DROPOUT,
                              kern_length=KERN_LENGTH,
                              F1=F1, D=D)
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

        # ── 8. Callbacks ───────────────────────────────────────────────
        cbs = [
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=patience,
                restore_best_weights=True, verbose=0),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5,
                patience=4, min_lr=1e-5, verbose=0)
        ]

        # ── 9. Train ───────────────────────────────────────────────────
        cw   = compute_class_weights(y_tr)
        hist = model.fit(
            X_tr_k, y_tr,
            validation_data=(X_val_k, y_val),
            epochs=max_epochs,
            batch_size=batch_size,
            class_weight=cw,
            callbacks=cbs,
            verbose=0
        )

        # ── 10. Predict test subject → majority vote ───────────────────
        ep_preds  = np.argmax(model.predict(X_te_k, verbose=0), axis=1)
        subj_pred = majority_vote(ep_preds)

        true_labels.append(true_label)
        pred_labels.append(subj_pred)
        histories.append({
            'loss':     hist.history['loss'],
            'val_loss': hist.history['val_loss'],
            'val_accuracy': hist.history.get('val_accuracy', []),
        })

        # ── 11. Free memory ────────────────────────────────────────────
        del X_tr, X_val, X_te, X_tr_k, X_val_k, X_te_k, model, hist
        gc.collect()
        tf.keras.backend.clear_session()

        fold_time = time.time() - t_fold
        fold_times.append(fold_time)

        # ── 12. Progress + ETA ─────────────────────────────────────────
        if (fold_i + 1) % verbose_every == 0 or fold_i == 0:
            avg_t  = np.mean(fold_times)
            eta    = (n_folds - fold_i - 1) * avg_t
            ep_run = len(histories[-1]['loss'])
            bval   = min(histories[-1]['val_loss'])
            lname  = label_names.get(true_label, str(true_label)) if label_names else str(true_label)
            pname  = label_names.get(subj_pred, str(subj_pred)) if label_names else str(subj_pred)
            print(f'  [{fold_i+1:3d}/{n_folds}] {test_sid} | '
                  f'true={lname} pred={pname} | '
                  f'epochs={ep_run} val_loss={bval:.4f} | '
                  f'fold={fold_time:.0f}s ETA={eta/60:.1f}min')

    true_labels = np.array(true_labels)
    pred_labels = np.array(pred_labels)

    return {
        'true':        true_labels,
        'pred':        pred_labels,
        'accuracy':    accuracy_score(true_labels, pred_labels),
        'f1_weighted': f1_score(true_labels, pred_labels,
                                 average='weighted', zero_division=0),
        'f1_macro':    f1_score(true_labels, pred_labels,
                                 average='macro', zero_division=0),
        'histories':   histories,
        'subject_ids': all_sids,
        'fold_times':  fold_times,
    }

print('LOSO function ready.')

LOSO function ready.


Cell 12 — Run Binary LOSO (AD vs HC)

In [13]:
print('=' * 60)
print('EEGNet — Binary LOSO (AD vs HC)')
print('Strict subject-level | Majority vote | No leakage')
print('=' * 60)
print()

t0 = time.time()
results_bin = run_eegnet_loso(
    subject_meta,
    n_classes=2,
    binary_remap={0: 0, 2: 1},   # AD->0, HC->1, FTD dropped
    label_names={0: 'AD', 1: 'HC'},
    verbose_every=5
)
elapsed = time.time() - t0

print()
print(f'Total time : {elapsed/60:.1f} min')
print(f'Accuracy   : {results_bin["accuracy"]*100:.2f}%')
print(f'F1 (W)     : {results_bin["f1_weighted"]*100:.2f}%')
print(f'F1 (M)     : {results_bin["f1_macro"]*100:.2f}%')
print()
print(classification_report(results_bin['true'], results_bin['pred'],
                             target_names=['AD', 'HC'], zero_division=0))

EEGNet — Binary LOSO (AD vs HC)
Strict subject-level | Majority vote | No leakage

LOSO: 65 folds | 2 classes
Epochs capped at 150/subject | batch=64 | max_train_epochs=50 | patience=10
Estimated time: 43–65 min on T4 GPU



/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'bou

  [  1/65] sub-001 | true=AD pred=HC | epochs=35 val_loss=0.1906 | fold=343s ETA=365.7min


/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'bou

/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'bou

...

  [ 55/65] sub-055 | true=HC pred=HC | epochs=18 val_loss=0.3841 | fold=191s ETA=35.0min


/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'bou

  [ 60/65] sub-060 | true=HC pred=AD | epochs=15 val_loss=0.3762 | fold=165s ETA=17.2min


/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'bou

  [ 65/65] sub-065 | true=HC pred=HC | epochs=24 val_loss=0.2434 | fold=242s ETA=0.0min

Total time : 225.9 min
Accuracy   : 78.46%
F1 (W)     : 78.49%
F1 (M)     : 78.46%

              precision    recall  f1-score   support

          AD       0.87      0.72      0.79        36
          HC       0.71      0.86      0.78        29

    accuracy                           0.78        65
   macro avg       0.79      0.79      0.78        65
weighted avg       0.80      0.78      0.78        65



Cell 13 — Run 3-class LOSO (AD vs FTD vs HC)

In [14]:
print('=' * 60)
print('EEGNet — 3-class LOSO (AD / FTD / HC)')
print('=' * 60)
print()

t0 = time.time()
results_3class = run_eegnet_loso(
    subject_meta,
    n_classes=3,
    binary_remap=None,
    label_names=LABEL_NAMES,
    verbose_every=5
)
elapsed = time.time() - t0

print()
print(f'Total time : {elapsed/60:.1f} min')
print(f'Accuracy   : {results_3class["accuracy"]*100:.2f}%')
print(f'F1 (W)     : {results_3class["f1_weighted"]*100:.2f}%')
print(f'F1 (M)     : {results_3class["f1_macro"]*100:.2f}%')
print()
print(classification_report(results_3class['true'], results_3class['pred'],
                             target_names=['AD', 'FTD', 'HC'], zero_division=0))

EEGNet — 3-class LOSO (AD / FTD / HC)

LOSO: 88 folds | 3 classes
Epochs capped at 150/subject | batch=64 | max_train_epochs=50 | patience=10
Estimated time: 59–88 min on T4 GPU



/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'bou

  [  1/88] sub-001 | true=AD pred=FTD | epochs=18 val_loss=1.0548 | fold=256s ETA=370.9min


/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'bou

  [  5/88] sub-005 | true=AD pred=AD | epochs=30 val_loss=1.0026 | fold=390s ETA=385.0min


...

/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'bou

  [ 70/88] sub-070 | true=FTD pred=HC | epochs=12 val_loss=1.0652 | fold=207s ETA=80.0min


/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'bou

  [ 75/88] sub-075 | true=FTD pred=FTD | epochs=21 val_loss=0.7826 | fold=311s ETA=58.0min


/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw    = mne.io.read_raw_eeglab(str(meta['path']),
/tmp/ipykernel_22/279953902.py:9: RuntimeWarning: The data contains 'bou

Cell 14 — Confusion matrices + training curve

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('white')
plt.rcParams.update({'font.size': 12})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Binary CM
ax = axes[0]
cm2 = confusion_matrix(results_bin['true'], results_bin['pred'], labels=[0,1])
sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['AD','HC'], yticklabels=['AD','HC'],
            cbar=False, linewidths=0.5)
ax.set_title(f'EEGNet Binary (AD vs HC)\n'
             f'Acc={results_bin["accuracy"]*100:.1f}%  '
             f'F1={results_bin["f1_weighted"]*100:.1f}%', fontweight='bold')
ax.set_ylabel('True'); ax.set_xlabel('Predicted')

# 3-class CM
ax2 = axes[1]
cm3 = confusion_matrix(results_3class['true'], results_3class['pred'], labels=[0,1,2])
sns.heatmap(cm3, annot=True, fmt='d', cmap='Greens', ax=ax2,
            xticklabels=['AD','FTD','HC'], yticklabels=['AD','FTD','HC'],
            cbar=False, linewidths=0.5)
ax2.set_title(f'EEGNet 3-class (AD/FTD/HC)\n'
              f'Acc={results_3class["accuracy"]*100:.1f}%  '
              f'F1={results_3class["f1_weighted"]*100:.1f}%', fontweight='bold')
ax2.set_ylabel('True'); ax2.set_xlabel('Predicted')

# Training curve (fold 0 representative)
ax3 = axes[2]
h   = results_bin['histories'][0]
ep  = range(1, len(h['loss']) + 1)
ax3.plot(ep, h['loss'],     label='Train loss', color='#3498DB', linewidth=2)
ax3.plot(ep, h['val_loss'], label='Val loss',   color='#E74C3C', linewidth=2)
ax3.set_title('Training Curve (Fold 1 — representative)', fontweight='bold')
ax3.set_xlabel('Epoch'); ax3.set_ylabel('Loss')
ax3.legend(); ax3.grid(True, alpha=0.3)

plt.tight_layout()
out_fig = RESULTS_DIR / 'fig05_eegnet_loso_results.png'
fig.savefig(str(out_fig), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_fig}')

Cell 15 — Save all results

In [ ]:
import json

# Fold time statistics
fold_t = results_bin['fold_times']
print(f'Fold time stats (binary): '
      f'mean={np.mean(fold_t):.0f}s  '
      f'min={np.min(fold_t):.0f}s  '
      f'max={np.max(fold_t):.0f}s')

summary = {
    'model': 'EEGNet (Lawhern et al. 2018)',
    'evaluation': 'Subject-level LOSO + majority vote — zero leakage',
    'dataset': 'ds004504 (OpenNeuro, 88 subjects)',
    'epoch_cap_per_subject': MAX_EPOCHS_PER_SUBJECT,
    'binary_AD_HC': {
        'n_subjects':  len(results_bin['subject_ids']),
        'accuracy':    round(float(results_bin['accuracy']), 4),
        'f1_weighted': round(float(results_bin['f1_weighted']), 4),
        'f1_macro':    round(float(results_bin['f1_macro']), 4),
        'total_time_min': round(sum(results_bin['fold_times'])/60, 1),
    },
    'multiclass_AD_FTD_HC': {
        'n_subjects':  len(results_3class['subject_ids']),
        'accuracy':    round(float(results_3class['accuracy']), 4),
        'f1_weighted': round(float(results_3class['f1_weighted']), 4),
        'f1_macro':    round(float(results_3class['f1_macro']), 4),
        'total_time_min': round(sum(results_3class['fold_times'])/60, 1),
    }
}

out_json = RESULTS_DIR / 'results_eegnet_loso.json'
with open(out_json, 'w') as f:
    json.dump(summary, f, indent=2)

# Per-subject predictions
pd.DataFrame({
    'subject_id': results_bin['subject_ids'],
    'true':       results_bin['true'],
    'pred':       results_bin['pred'],
    'correct':    (results_bin['true'] == results_bin['pred']).astype(int)
}).to_csv(str(RESULTS_DIR / 'eegnet_predictions_binary.csv'), index=False)

pd.DataFrame({
    'subject_id': results_3class['subject_ids'],
    'true':       results_3class['true'],
    'pred':       results_3class['pred'],
    'correct':    (results_3class['true'] == results_3class['pred']).astype(int)
}).to_csv(str(RESULTS_DIR / 'eegnet_predictions_3class.csv'), index=False)

print()
print('=== EEGNet LOSO Complete ===')
print()
print(f"Binary  AD vs HC  | Acc: {summary['binary_AD_HC']['accuracy']*100:.2f}%"
      f" | F1w: {summary['binary_AD_HC']['f1_weighted']*100:.2f}%"
      f" | Time: {summary['binary_AD_HC']['total_time_min']} min")
print(f"3-class AD/FTD/HC | Acc: {summary['multiclass_AD_FTD_HC']['accuracy']*100:.2f}%"
      f" | F1w: {summary['multiclass_AD_FTD_HC']['f1_weighted']*100:.2f}%"
      f" | Time: {summary['multiclass_AD_FTD_HC']['total_time_min']} min")
print()
print(f'Results saved to: {RESULTS_DIR}')
print()
print('Next: 05_sir_eegnet_loso.ipynb')